In [ ]:
# CPU / BLAS thread controls
# Edit CPU_AFFINITY_RANGE and/or CPU_CAP before running the rest of the notebook.
import os
import multiprocessing as mp
from pathlib import Path

CPU_AFFINITY_RANGE = (90, 110)
CPU_CAP = 14

cpu_available = mp.cpu_count()
if os.environ.get("GENTRG_SMOKE_TEST") == "1":
    CPU_CAP = min(CPU_CAP, 4)

if CPU_AFFINITY_RANGE is not None:
    cpu_start, cpu_stop = [int(v) for v in CPU_AFFINITY_RANGE]
    if cpu_stop < cpu_start:
        raise ValueError(f"CPU_AFFINITY_RANGE must satisfy stop >= start, got {CPU_AFFINITY_RANGE}")
    selected_cpus = list(range(cpu_start, cpu_stop + 1))[: int(CPU_CAP)]
else:
    selected_cpus = None

cpu_use = len(selected_cpus) if selected_cpus is not None else max(1, min(int(CPU_CAP), cpu_available))
os.environ["MY_CPU_COUNT"] = str(cpu_use)
os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "mplconfig_scale_invariant_gaussian_mera"))
for _var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMEXPR_MAX_THREADS",
):
    os.environ[_var] = str(cpu_use)

if selected_cpus is not None:
    try:
        os.sched_setaffinity(0, set(selected_cpus))
        print(f"CPU affinity set to cores [{min(selected_cpus)}, {max(selected_cpus)}]")
    except Exception as exc:
        print(f"CPU affinity not set: {exc}")

print(f"cpu_available={cpu_available}, cpu_use={cpu_use}, CPU_AFFINITY_RANGE={CPU_AFFINITY_RANGE}")


# Scale-Invariant Gaussian MERA Prototype

This notebook constructs the same half-filled OW flattened-H domain-wall state as `gaussian_entanglement_rg.ipynb`, but now tests a repeated binary Gaussian MERA layer along the domain-wall direction:

\[
C' = W^\dagger U C U^\dagger W.
\]

The default run uses `Nx=20`, `Ny=40`, `nshell=1`, `alpha_1=1`, `alpha_2=30`, `DW=True`, and `dw_truncation=True`. Set `GENTRG_SMOKE_TEST=1` before execution for a reduced validation run.

## Setup

In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from scipy.linalg import expm
from scipy.optimize import minimize


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = ROOT / "notebooks" / "flattened_hamiltonian_analysis" / "gaussian_entanglement_rg"
SMOKE_TEST = os.environ.get("GENTRG_SMOKE_TEST") == "1"
DATA_DIR = NOTEBOOK_DIR / "data" / "mera" / ("smoke" if SMOKE_TEST else "")
FIGURES_DIR = NOTEBOOK_DIR / "figures" / "mera" / ("smoke" if SMOKE_TEST else "")
SRC = ROOT / "src"

for path in (DATA_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

classA_U1FGTN_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module)
classA_U1FGTN = classA_U1FGTN_module.classA_U1FGTN

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if "CMU Sans Serif" in available_fonts:
    plt.rcParams["font.family"] = "CMU Sans Serif"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
    print("CMU Sans Serif not found; using DejaVu Sans")

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "lines.linewidth": 1.3,
    "mathtext.fontset": "cm",
})
np.set_printoptions(precision=6, suppress=True)

print(f"ROOT = {ROOT}")
print(f"NOTEBOOK_DIR = {NOTEBOOK_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"FIGURES_DIR = {FIGURES_DIR}")


## Run Controls

In [ ]:
NX = 20
NY = 40
NSHELL = 1
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW = True
DW_TRUNCATION = True
TRIAL_ORBITAL = "X"

BASE_FIT_AY_MIN = 8
MIN_COARSE_NY = 10
ENTROPY_CLIP_EPS = 1e-12
DOMAIN_WALL_PROFILE_WIDTH = 1
R_MAX = 4
LOSS_EPS = 1e-12
RUN_STAGE2_OPTIMIZATION = True
ACTIVE_DIM = 12
OPT_MAXITER = 6
OPT_MAXFUN = 180
OPT_PARAM_SCALE = 0.04

if SMOKE_TEST:
    NX = 6
    NY = 16
    BASE_FIT_AY_MIN = 2
    MIN_COARSE_NY = 8
    R_MAX = 2
    ACTIVE_DIM = 4
    OPT_MAXITER = 4
    OPT_MAXFUN = 80
    OPT_PARAM_SCALE = 0.05

Q_ORBITALS = 2 * NX
HALF_FILLING = Q_ORBITALS * NY // 2

print(f"SMOKE_TEST = {SMOKE_TEST}")
print(f"Nx, Ny = {NX}, {NY}")
print(f"q = 2*Nx = {Q_ORBITALS}")
print(f"half filling = {HALF_FILLING} / {Q_ORBITALS * NY}")
print(f"nshell = {NSHELL}, alpha_1 = {ALPHA_1}, alpha_2 = {ALPHA_2}")
print(f"R_MAX = {R_MAX}, ACTIVE_DIM = {ACTIVE_DIM}, RUN_STAGE2_OPTIMIZATION = {RUN_STAGE2_OPTIMIZATION}")


## State and Entropy Helpers

In [ ]:
def build_flattened_hamiltonian(model, trial_orbital=TRIAL_ORBITAL, dw_truncation=DW_TRUNCATION):
    nlayer = 2 * model.Nx * model.Ny
    model.construct_OW_projectors(
        nshell=model.nshell,
        DW=model.DW,
        trial_orbitals=trial_orbital,
        dw_truncation=dw_truncation,
    )
    WA_p = model.WF_Ap.reshape(nlayer, -1)
    WB_p = model.WF_Bp.reshape(nlayer, -1)
    WA_m = model.WF_Am.reshape(nlayer, -1)
    WB_m = model.WF_Bm.reshape(nlayer, -1)
    H_flat = WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T
    return 0.5 * (H_flat + H_flat.conj().T)


def build_half_filled_flattened_state(nx=NX, ny=NY):
    model = classA_U1FGTN(nx, ny, nshell=NSHELL, DW=DW, alpha_1=ALPHA_1, alpha_2=ALPHA_2)
    H_flat = build_flattened_hamiltonian(model)
    evals, evecs = np.linalg.eigh(H_flat)
    nlayer = H_flat.shape[0]
    n_occ = nlayer // 2
    occ_mask = evals < 0.0
    neg_count = int(np.count_nonzero(occ_mask))
    fallback_used = False
    if neg_count != n_occ:
        occ_mask = np.zeros_like(evals, dtype=bool)
        occ_mask[np.argsort(evals)[:n_occ]] = True
        fallback_used = True
    U_occ = evecs[:, occ_mask]
    C = U_occ @ U_occ.conj().T
    C = 0.5 * (C + C.conj().T)
    C_pm1 = 2.0 * C.conj() - np.eye(nlayer, dtype=np.complex128)
    return {
        "model": model,
        "H_flat": H_flat,
        "evals": np.real_if_close(evals),
        "C": C,
        "C_pm1": C_pm1,
        "n_occ": int(n_occ),
        "neg_count": neg_count,
        "fallback_used": bool(fallback_used),
        "spectral_gap": float(np.min(np.abs(evals))),
        "mean_abs_eval": float(np.mean(np.abs(evals))),
    }


def interval_indices(n_sites, q, length, start=0):
    n_sites = int(n_sites)
    q = int(q)
    length = int(length)
    start = int(start) % n_sites
    if length < 0 or length > n_sites:
        raise ValueError(f"length must lie in [0, {n_sites}], got {length}")
    rows = (np.arange(length, dtype=int) + start) % n_sites
    return np.concatenate([np.arange(row * q, (row + 1) * q, dtype=int) for row in rows])


def entropy_from_correlation_submatrix(C_sub, clip_eps=ENTROPY_CLIP_EPS):
    C_sub = np.asarray(C_sub, dtype=np.complex128)
    C_sub = 0.5 * (C_sub + C_sub.conj().T)
    lambdas = np.linalg.eigvalsh(C_sub)
    lambdas = np.clip(np.real_if_close(lambdas), clip_eps, 1.0 - clip_eps)
    return float(-np.sum(lambdas * np.log(lambdas) + (1.0 - lambdas) * np.log(1.0 - lambdas)))


def entropy_for_interval(C, n_sites, q, length, start=0):
    if int(length) == 0:
        return 0.0
    idx = interval_indices(n_sites, q, length, start=start)
    return entropy_from_correlation_submatrix(C[np.ix_(idx, idx)])


def mean_entropy_profile(C, n_sites, q, max_length=None):
    n_sites = int(n_sites)
    if max_length is None:
        max_length = n_sites // 2
    lengths = np.arange(1, int(max_length) + 1, dtype=int)
    values = []
    for length in lengths:
        entropies = [entropy_for_interval(C, n_sites, q, length, start=y0) for y0 in range(n_sites)]
        values.append(float(np.mean(entropies)))
    return {"Ay_values": lengths, "avg_entropy": np.asarray(values, dtype=float)}


def logchord_values(Ay_values, ny):
    Ay_values = np.asarray(Ay_values, dtype=float)
    return np.log((float(ny) / np.pi) * np.sin(np.pi * Ay_values / float(ny)))


def fit_ay_min_for_ny(ny):
    ny = int(ny)
    if ny >= 32:
        return int(BASE_FIT_AY_MIN)
    return max(2, int(ny // 5))


def fit_entropy_logchord(Ay_values, entropy_values, ny, fit_ay_min=None):
    Ay_values = np.asarray(Ay_values, dtype=int)
    entropy_values = np.asarray(entropy_values, dtype=float)
    if fit_ay_min is None:
        fit_ay_min = fit_ay_min_for_ny(ny)
    x = logchord_values(Ay_values, ny)
    fit_mask = (Ay_values >= int(fit_ay_min)) & (Ay_values <= int(ny // 2)) & np.isfinite(x) & np.isfinite(entropy_values)
    if int(np.count_nonzero(fit_mask)) < 3:
        raise ValueError(f"Need at least 3 fit points for Ny={ny}, fit_ay_min={fit_ay_min}")
    x_fit = x[fit_mask]
    y_fit = entropy_values[fit_mask]
    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_hat = slope * x_fit + intercept
    ss_res = float(np.sum((y_fit - y_hat) ** 2))
    ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
    x_centered = x_fit - np.mean(x_fit)
    sxx = float(np.sum(x_centered ** 2))
    sigma2 = ss_res / float(len(x_fit) - 2) if len(x_fit) > 2 else np.nan
    slope_err = float(np.sqrt(max(sigma2, 0.0) / sxx)) if sxx > 0 and np.isfinite(sigma2) else np.nan
    r2 = 1.0 if np.isclose(ss_tot, 0.0) and np.isclose(ss_res, 0.0) else 1.0 - ss_res / ss_tot
    return {
        "slope": float(slope),
        "slope_err": float(slope_err),
        "intercept": float(intercept),
        "r2": float(r2),
        "c_est": float(3.0 * slope),
        "c_est_err": float(3.0 * slope_err) if np.isfinite(slope_err) else np.nan,
        "log_chord": x,
        "fit_mask": fit_mask,
        "fit_line": slope * x + intercept,
        "fit_ay_min": int(fit_ay_min),
    }


def entanglement_spectrum(C, n_sites, q, length=None, start=0):
    if length is None:
        length = n_sites // 2
    idx = interval_indices(n_sites, q, int(length), start=start)
    C_sub = C[np.ix_(idx, idx)]
    C_sub = 0.5 * (C_sub + C_sub.conj().T)
    return np.linalg.eigvalsh(C_sub)


## Repeated Gaussian MERA Layer

In [ ]:
def row_indices(row, q, n_sites):
    row = int(row) % int(n_sites)
    return np.arange(row * q, (row + 1) * q, dtype=int)


def pair_indices(row_a, row_b, q, n_sites):
    return np.concatenate([row_indices(row_a, q, n_sites), row_indices(row_b, q, n_sites)])


def averaged_pair_correlation(C, n_sites, q, pairs):
    acc = np.zeros((2 * q, 2 * q), dtype=np.complex128)
    for row_a, row_b in pairs:
        idx = pair_indices(row_a, row_b, q, n_sites)
        acc += C[np.ix_(idx, idx)]
    acc /= float(len(pairs))
    return 0.5 * (acc + acc.conj().T)


def block_pairs(n_sites):
    return [(2 * j, 2 * j + 1) for j in range(int(n_sites) // 2)]


def boundary_pairs(n_sites):
    return [(2 * j + 1, (2 * j + 2) % int(n_sites)) for j in range(int(n_sites) // 2)]


def init_repeated_isometry_from_average(C, n_sites, q):
    C_block = averaged_pair_correlation(C, n_sites, q, block_pairs(n_sites))
    occ, vecs = np.linalg.eigh(C_block)
    occ = np.clip(np.real_if_close(occ), 0.0, 1.0)
    chi = occ * (1.0 - occ)
    keep = np.argsort(chi)[-q:]
    discard = np.setdiff1d(np.arange(2 * q), keep, assume_unique=False)
    keep = keep[np.argsort(occ[keep])]
    w = vecs[:, keep]
    return {
        "w": w,
        "kept_occ": occ[keep].copy(),
        "discarded_occ": occ[discard].copy(),
        "kept_chi": chi[keep].copy(),
        "discarded_chi": chi[discard].copy(),
        "average_block_C": C_block,
    }


def apply_repeated_boundary_unitary(C, n_sites, q, u):
    if u is None:
        return np.asarray(C, dtype=np.complex128)
    C_u = np.asarray(C, dtype=np.complex128).copy()
    udag = u.conj().T
    for row_a, row_b in boundary_pairs(n_sites):
        idx = pair_indices(row_a, row_b, q, n_sites)
        C_u[idx, :] = u @ C_u[idx, :]
    for row_a, row_b in boundary_pairs(n_sites):
        idx = pair_indices(row_a, row_b, q, n_sites)
        C_u[:, idx] = C_u[:, idx] @ udag
    return 0.5 * (C_u + C_u.conj().T)


def coarsegrain_with_repeated_isometry(C, n_sites, q, w):
    n_sites = int(n_sites)
    q = int(q)
    n_blocks = n_sites // 2
    C_next = np.zeros((n_blocks * q, n_blocks * q), dtype=np.complex128)
    for a in range(n_blocks):
        ia = pair_indices(2 * a, 2 * a + 1, q, n_sites)
        oa = row_indices(a, q, n_blocks)
        for b in range(n_blocks):
            ib = pair_indices(2 * b, 2 * b + 1, q, n_sites)
            ob = row_indices(b, q, n_blocks)
            C_next[np.ix_(oa, ob)] = w.conj().T @ C[np.ix_(ia, ib)] @ w
    return 0.5 * (C_next + C_next.conj().T)


def repeated_isometry_global_matrix(n_sites, q, w):
    n_sites = int(n_sites)
    q = int(q)
    n_blocks = n_sites // 2
    W = np.zeros((n_sites * q, n_blocks * q), dtype=np.complex128)
    for block in range(n_blocks):
        idx = pair_indices(2 * block, 2 * block + 1, q, n_sites)
        out = row_indices(block, q, n_blocks)
        W[np.ix_(idx, out)] = w
    return W


def mera_layer(C, n_sites, q, w, u=None):
    C_u = apply_repeated_boundary_unitary(C, n_sites, q, u)
    return coarsegrain_with_repeated_isometry(C_u, n_sites, q, w)


def average_block_correlator(C, n_sites, q, r):
    n_sites = int(n_sites)
    q = int(q)
    r = int(r) % n_sites
    acc = np.zeros((q, q), dtype=np.complex128)
    for y in range(n_sites):
        i = row_indices(y, q, n_sites)
        j = row_indices(y + r, q, n_sites)
        acc += C[np.ix_(i, j)]
    return acc / float(n_sites)


def fixed_point_loss(C_target, C_coarse, n_sites_target, n_sites_coarse, q, r_max=R_MAX, return_rows=False, scale_factor=2):
    max_r = min(int(r_max), int(n_sites_coarse // 2))
    total = 0.0
    rows = []
    for r in range(max_r + 1):
        coarse = average_block_correlator(C_coarse, n_sites_coarse, q, r)
        target = average_block_correlator(C_target, n_sites_target, q, int(scale_factor) * r)
        numerator = float(np.linalg.norm(coarse - target, ord="fro") ** 2)
        denom = float(np.linalg.norm(target, ord="fro") ** 2 + LOSS_EPS)
        weight = 1.0 / float(1 + r)
        contribution = weight * numerator / denom
        total += contribution
        rows.append({
            "r": int(r),
            "weight": float(weight),
            "numerator": numerator,
            "denominator": denom,
            "relative_mismatch": float(numerator / denom),
            "weighted_contribution": float(contribution),
        })
    if return_rows:
        return float(total), rows
    return float(total)


def original_x_profile_from_embedding(embedding, nx, ny0):
    embedding = np.asarray(embedding, dtype=np.complex128)
    weights_by_original_mode = np.sum(np.abs(embedding) ** 2, axis=1)
    profile = np.zeros(int(nx), dtype=float)
    for y in range(int(ny0)):
        for x in range(int(nx)):
            base = 2 * int(nx) * y + 2 * x
            profile[x] += float(weights_by_original_mode[base] + weights_by_original_mode[base + 1])
    total = float(np.sum(profile))
    if total > 0.0:
        profile /= total
    return profile


def domain_wall_x_mask(nx, dw_locs, width=DOMAIN_WALL_PROFILE_WIDTH):
    mask = np.zeros(int(nx), dtype=bool)
    for loc in dw_locs:
        loc = int(loc) % int(nx)
        for delta in range(-int(width), int(width) + 1):
            mask[(loc + delta) % int(nx)] = True
    return mask


## Active-Subspace Disentangler

In [ ]:
def init_active_boundary_basis(C, n_sites, q, active_dim):
    C_boundary = averaged_pair_correlation(C, n_sites, q, boundary_pairs(n_sites))
    occ, vecs = np.linalg.eigh(C_boundary)
    occ = np.clip(np.real_if_close(occ), 0.0, 1.0)
    chi = occ * (1.0 - occ)
    active = np.argsort(chi)[-int(active_dim):]
    active = active[np.argsort(occ[active])]
    basis = vecs[:, active]
    return {
        "basis": basis,
        "active_occ": occ[active].copy(),
        "active_chi": chi[active].copy(),
        "average_boundary_C": C_boundary,
    }


def unpack_antihermitian(params, m, scale=OPT_PARAM_SCALE):
    params = np.asarray(params, dtype=float) * float(scale)
    K = np.zeros((m, m), dtype=np.complex128)
    cursor = 0
    for i in range(m):
        K[i, i] = 1j * params[cursor]
        cursor += 1
    for i in range(m):
        for j in range(i + 1, m):
            a = params[cursor]
            b = params[cursor + 1]
            cursor += 2
            K[i, j] = a + 1j * b
            K[j, i] = -a + 1j * b
    return K


def n_antihermitian_params(m):
    return int(m * m)


def active_unitary_from_params(params, basis, q):
    basis = np.asarray(basis, dtype=np.complex128)
    m = basis.shape[1]
    K = unpack_antihermitian(params, m)
    u_active = expm(K)
    eye = np.eye(2 * int(q), dtype=np.complex128)
    u = eye + basis @ (u_active - np.eye(m, dtype=np.complex128)) @ basis.conj().T
    return 0.5 * (u + np.linalg.inv(u).conj().T)


def optimize_active_disentangler(C_target, n_sites, q, w, basis, r_max=R_MAX):
    n_params = n_antihermitian_params(basis.shape[1])
    x0 = np.zeros(n_params, dtype=float)

    def objective(x):
        u = active_unitary_from_params(x, basis, q)
        C_next = mera_layer(C_target, n_sites, q, w, u=u)
        return fixed_point_loss(C_target, C_next, n_sites, n_sites // 2, q, r_max=r_max)

    initial_loss = objective(x0)
    result = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        options={"maxiter": int(OPT_MAXITER), "maxfun": int(OPT_MAXFUN), "ftol": 1e-10, "gtol": 1e-6},
    )
    u_opt = active_unitary_from_params(result.x, basis, q)
    final_loss = objective(result.x)
    return {
        "result": result,
        "u": u_opt,
        "initial_loss": float(initial_loss),
        "final_loss": float(final_loss),
    }


## Build Target State and Initialize Repeated Tensors

In [ ]:
state = build_half_filled_flattened_state(NX, NY)
model = state["model"]
C0 = state["C"]
DW_LOCS = list(getattr(model, "DW_loc", []))

init_w = init_repeated_isometry_from_average(C0, NY, Q_ORBITALS)
w0 = init_w["w"]
u_identity = np.eye(2 * Q_ORBITALS, dtype=np.complex128)
active = init_active_boundary_basis(C0, NY, Q_ORBITALS, ACTIVE_DIM)

print(f"DW locations = {DW_LOCS}")
print(f"H_flat shape = {state['H_flat'].shape}")
print(f"negative eigenvalue count = {state['neg_count']}")
print(f"filled modes = {state['n_occ']}")
print(f"fallback_used = {state['fallback_used']}")
print(f"spectral_gap = {state['spectral_gap']:.6e}")
print(f"projector idempotency ||C^2-C||_F = {np.linalg.norm(C0 @ C0 - C0):.6e}")
print(f"isometry error ||w^†w-I||_F = {np.linalg.norm(w0.conj().T @ w0 - np.eye(Q_ORBITALS)):.6e}")
print(f"active basis error ||B^†B-I||_F = {np.linalg.norm(active['basis'].conj().T @ active['basis'] - np.eye(ACTIVE_DIM)):.6e}")


## Stage 1: Repeated-Isometry Baseline

In [ ]:
C_stage1 = mera_layer(C0, NY, Q_ORBITALS, w0, u=u_identity)
loss_stage1, mismatch_stage1_rows = fixed_point_loss(
    C0, C_stage1, NY, NY // 2, Q_ORBITALS, r_max=R_MAX, return_rows=True
)
print(f"Stage 1 fixed-point loss = {loss_stage1:.8e}")


## Stage 2: Active-Subspace Disentangler Optimization

In [ ]:
if RUN_STAGE2_OPTIMIZATION:
    opt = optimize_active_disentangler(C0, NY, Q_ORBITALS, w0, active["basis"], r_max=R_MAX)
    u_opt = opt["u"]
    C_stage2 = mera_layer(C0, NY, Q_ORBITALS, w0, u=u_opt)
    loss_stage2, mismatch_stage2_rows = fixed_point_loss(
        C0, C_stage2, NY, NY // 2, Q_ORBITALS, r_max=R_MAX, return_rows=True
    )
    print(f"Stage 2 initial loss = {opt['initial_loss']:.8e}")
    print(f"Stage 2 final loss   = {loss_stage2:.8e}")
    print(f"optimizer success = {bool(opt['result'].success)}")
    print(f"optimizer message = {opt['result'].message}")
    print(f"function evals = {int(opt['result'].nfev)}")
else:
    opt = None
    u_opt = u_identity
    C_stage2 = C_stage1
    loss_stage2 = loss_stage1
    mismatch_stage2_rows = mismatch_stage1_rows


## Repeated MERA Flow Diagnostics

In [ ]:
def analyze_state(label, C, n_sites, q, embedding, loss_value=np.nan):
    entropy_data = mean_entropy_profile(C, n_sites, q)
    fit = fit_entropy_logchord(entropy_data["Ay_values"], entropy_data["avg_entropy"], n_sites)
    x_profile = original_x_profile_from_embedding(embedding, NX, NY)
    dw_mask = domain_wall_x_mask(NX, DW_LOCS)
    eigs = entanglement_spectrum(C, n_sites, q)
    return {
        "label": str(label),
        "C": C,
        "n_sites": int(n_sites),
        "q": int(q),
        "embedding": embedding,
        "fit": fit,
        "entropy_data": entropy_data,
        "x_profile": x_profile,
        "dw_weight": float(np.sum(x_profile[dw_mask])),
        "entanglement_spectrum": eigs,
        "fixed_point_loss": float(loss_value),
    }


def repeated_flow(C_start, n_sites_start, q, w, u, max_layers=2):
    items = []
    C = C_start
    n_sites = int(n_sites_start)
    embedding = np.eye(C.shape[0], dtype=np.complex128)
    items.append(analyze_state("target", C, n_sites, q, embedding, loss_value=0.0))
    for layer in range(1, int(max_layers) + 1):
        C = mera_layer(C, n_sites, q, w, u=u)
        W = repeated_isometry_global_matrix(n_sites, q, w)
        embedding = embedding @ W
        n_sites //= 2
        loss = fixed_point_loss(C_start, C, n_sites_start, n_sites, q, r_max=min(R_MAX, n_sites // 2), scale_factor=2 ** layer)
        items.append(analyze_state(f"mera_layer_{layer}", C, n_sites, q, embedding, loss_value=loss))
        if n_sites <= MIN_COARSE_NY:
            break
    return items


stage1_flow = repeated_flow(C0, NY, Q_ORBITALS, w0, u_identity, max_layers=2)
stage2_flow = repeated_flow(C0, NY, Q_ORBITALS, w0, u_opt, max_layers=2)

summary_rows = []
for stage, flow in [("stage1_isometry", stage1_flow), ("stage2_active_u", stage2_flow)]:
    for layer, item in enumerate(flow):
        summary_rows.append({
            "smoke_test": bool(SMOKE_TEST),
            "stage": stage,
            "layer": int(layer),
            "Nx": int(NX),
            "Ny_initial": int(NY),
            "Ny_current": int(item["n_sites"]),
            "q_orbitals": int(item["q"]),
            "active_dim": int(ACTIVE_DIM),
            "fixed_point_loss": float(item["fixed_point_loss"]),
            "slope": item["fit"]["slope"],
            "slope_err": item["fit"]["slope_err"],
            "c_est": item["fit"]["c_est"],
            "c_est_err": item["fit"]["c_est_err"],
            "r2": item["fit"]["r2"],
            "domain_wall_x_weight_width1": item["dw_weight"],
        })
summary_df = pd.DataFrame(summary_rows)
summary_path = DATA_DIR / "scale_invariant_gaussian_mera_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary_df


In [ ]:
mismatch_df = pd.concat([
    pd.DataFrame(mismatch_stage1_rows).assign(stage="stage1_isometry"),
    pd.DataFrame(mismatch_stage2_rows).assign(stage="stage2_active_u"),
], ignore_index=True)
mismatch_path = DATA_DIR / "scale_invariant_gaussian_mera_mismatch.csv"
mismatch_df.to_csv(mismatch_path, index=False)
print(f"Saved mismatch data: {mismatch_path}")
mismatch_df


In [ ]:
occupation_rows = []
for sector, occ, chi in [
    ("kept", init_w["kept_occ"], init_w["kept_chi"]),
    ("discarded", init_w["discarded_occ"], init_w["discarded_chi"]),
]:
    for mode, (n, c) in enumerate(zip(occ, chi)):
        occupation_rows.append({"sector": sector, "mode": int(mode), "occupation": float(n), "chi": float(c)})
occupation_df = pd.DataFrame(occupation_rows)
occupation_path = DATA_DIR / "scale_invariant_gaussian_mera_occupations.csv"
occupation_df.to_csv(occupation_path, index=False)
print(f"Saved occupation spectra: {occupation_path}")
occupation_df.head()


In [ ]:
profile_rows = []
for stage, flow in [("stage1_isometry", stage1_flow), ("stage2_active_u", stage2_flow)]:
    for layer, item in enumerate(flow):
        for x, weight in enumerate(item["x_profile"]):
            profile_rows.append({"stage": stage, "layer": int(layer), "x": int(x), "weight": float(weight)})
profile_df = pd.DataFrame(profile_rows)
profile_path = DATA_DIR / "scale_invariant_gaussian_mera_x_profiles.csv"
profile_df.to_csv(profile_path, index=False)
print(f"Saved x profiles: {profile_path}")
profile_df.head()


In [ ]:
spectrum_rows = []
for stage, flow in [("stage1_isometry", stage1_flow), ("stage2_active_u", stage2_flow)]:
    for layer, item in enumerate(flow):
        eigs = np.asarray(item["entanglement_spectrum"], dtype=float)
        order = np.argsort(np.abs(eigs - 0.5))
        for rank, idx in enumerate(order[: min(80, len(order))]):
            spectrum_rows.append({
                "stage": stage,
                "layer": int(layer),
                "rank_near_half": int(rank),
                "lambda": float(eigs[idx]),
                "abs_lambda_minus_half": float(abs(eigs[idx] - 0.5)),
            })
spectrum_df = pd.DataFrame(spectrum_rows)
spectrum_path = DATA_DIR / "scale_invariant_gaussian_mera_entanglement_spectrum.csv"
spectrum_df.to_csv(spectrum_path, index=False)
print(f"Saved entanglement spectra: {spectrum_path}")
spectrum_df.head()


## Fixed-Point Loss

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
loss_plot = pd.DataFrame([
    {"stage": "stage1", "loss": loss_stage1},
    {"stage": "stage2", "loss": loss_stage2},
])
ax.bar(loss_plot["stage"], loss_plot["loss"], color=["tab:blue", "tab:green"])
ax.set_ylabel(r"fixed-point loss $\mathcal{L}$")
ax.set_title("MERA fixed-point objective")
ax.grid(axis="y", alpha=0.25, linewidth=0.4)
fig.tight_layout()
path = FIGURES_DIR / "fixed_point_loss.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Correlator Mismatch Versus Separation

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
for stage, plot_df in mismatch_df.groupby("stage"):
    ax.plot(plot_df["r"], plot_df["relative_mismatch"], marker="o", label=stage)
ax.set_xlabel(r"coarse separation $r$")
ax.set_ylabel("relative mismatch")
ax.set_title("Block-correlator fixed-point mismatch")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES_DIR / "correlator_mismatch_vs_r.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Central Charge Under Repeated MERA Layers

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.45))
for stage, plot_df in summary_df.groupby("stage"):
    ax.errorbar(plot_df["layer"], plot_df["c_est"], yerr=plot_df["c_est_err"], marker="o", capsize=2, label=stage)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1.0, label=r"$c_{\rm eff}=1$")
ax.set_xlabel("MERA layer")
ax.set_ylabel(r"$c_{\rm est}=3m$")
ax.set_title("Entropy coefficient under repeated MERA")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES_DIR / "central_charge_mera_layers.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Natural-Orbital Kept and Discarded Spectra

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
for sector, color, alpha in [("discarded", "tab:gray", 0.45), ("kept", "tab:blue", 0.75)]:
    vals = np.sort(occupation_df[occupation_df["sector"] == sector]["occupation"].to_numpy())
    ax.scatter(np.arange(vals.size), vals, s=8, color=color, alpha=alpha, label=sector)
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("mode rank")
ax.set_ylabel(r"natural occupation $n_a$")
ax.set_title("Repeated-isometry initialization")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES_DIR / "natural_orbital_init_spectrum.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Retained `x` Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.45), sharey=True)
for ax, stage in zip(axes, ["stage1_isometry", "stage2_active_u"]):
    plot_df = profile_df[profile_df["stage"] == stage]
    for layer, layer_df in plot_df.groupby("layer"):
        ax.plot(layer_df["x"], layer_df["weight"], marker="o", markersize=2.5, label=rf"layer {layer}")
    for dw in DW_LOCS:
        ax.axvline(dw, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
    ax.set_title(stage)
    ax.set_xlabel(r"original $x$")
    ax.grid(alpha=0.25, linewidth=0.4)
    ax.legend(frameon=False)
axes[0].set_ylabel("retained-subspace weight")
fig.tight_layout()
path = FIGURES_DIR / "retained_x_profile_mera.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Entanglement Spectrum Near `1/2`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.45), sharey=True)
for ax, stage in zip(axes, ["stage1_isometry", "stage2_active_u"]):
    plot_df = spectrum_df[spectrum_df["stage"] == stage]
    for layer, layer_df in plot_df.groupby("layer"):
        near = layer_df.sort_values("rank_near_half").head(40)
        ax.scatter(np.full(len(near), layer, dtype=float), near["lambda"], s=8, alpha=0.65, label=rf"layer {layer}")
    ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(stage)
    ax.set_xlabel("MERA layer")
    ax.grid(alpha=0.25, linewidth=0.4)
    ax.legend(frameon=False)
axes[0].set_ylabel(r"entanglement eigenvalue $\lambda$")
fig.tight_layout()
path = FIGURES_DIR / "entanglement_spectrum_mera.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Interpretation

This is a practical scale-invariant Gaussian MERA prototype. Stage 1 tests whether a single repeated isometry learned from the average two-row natural-orbital problem already behaves like a fixed-point coarse grainer. Stage 2 adds a bounded active-subspace boundary disentangler. The fixed-point loss is gauge-sensitive because the coarse supersite basis is the retained natural-orbital basis; therefore the entropy, entanglement spectrum, and DW-localized retained profile should be interpreted alongside the loss rather than replaced by it.